# 02 · Classical ML: purchase prediction and customer segmentation

**Supervised target: `purchase_flag`.** It answers the core merchandising question *"will this product view convert?"*. It is available for every interaction, it is balanced (51/49), and a conversion score can rank which shoppers to nudge (a coupon or reminder) and which products need better content.

In [ ]:
# --- setup: make the project package importable and show the run mode ---
import sys, os, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import src                                   # sets macOS OpenMP compatibility flags
from src import config
from src.llm import mode_label
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", palette="Set2"); plt.rcParams["figure.dpi"] = 110
OUT = config.OUTPUTS
print("Mode:", mode_label(), "| outputs ->", OUT.relative_to(ROOT))

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict, cross_val_score
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.inspection import permutation_importance
from src import data_prep, ml
df = data_prep.load_clean()

## Part A: Supervised learning

### A1. Features and deliberate exclusions

In [ ]:
print("numerical :", ml.NUM_FEATURES); print("binary    :", ml.BIN_FEATURES); print("categorical:", ml.CAT_FEATURES)
pd.Series(ml.EXCLUDED, name="reason excluded").to_frame()

**Preprocessing pipeline** (`ml.preprocessor`), fitted inside each training fold so nothing leaks:
- **numerical:** median imputation → `StandardScaler`. Scaling is applied for the linear model only; tree models skip it.
- **binary:** most-frequent imputation.
- **categorical:** constant `Unknown` imputation → `OneHotEncoder(handle_unknown="ignore", min_frequency=10)`. Rare levels are grouped, and unseen levels are ignored at prediction time.

**Class imbalance:** none to speak of (51% / 49%), so no resampling or class weights are used. PR-AUC is reported next to ROC-AUC anyway.

### A2. Split: hold-out test + cross-validated tuning

In [ ]:
X = df[ml.NUM_FEATURES + ml.BIN_FEATURES + ml.CAT_FEATURES]; y = df[ml.TARGET]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=config.SEED)
cv = StratifiedKFold(5, shuffle=True, random_state=config.SEED)
print(f"train {X_tr.shape}, test {X_te.shape}; purchase rate train {y_tr.mean():.3f} / test {y_te.mean():.3f}")

In [ ]:
searches, rows = {}, []
for name, (pipe, grid) in ml.candidate_models().items():
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="roc_auc", n_jobs=-1).fit(X_tr, y_tr)
    searches[name] = gs
    rows.append({"model": name, "CV ROC-AUC": gs.best_score_, "CV std": gs.cv_results_["std_test_score"][gs.best_index_],
                 "best params": {k.replace("clf__", ""): v for k, v in gs.best_params_.items()}})
cv_table = pd.DataFrame(rows).set_index("model").round(3)
cv_table.to_csv(OUT / "02_cv_results.csv"); cv_table

### A3. Test-set evaluation

In [ ]:
test_rows, probs = [], {}
for name, gs in searches.items():
    p = gs.predict_proba(X_te)[:, 1]; probs[name] = p; pred = (p >= 0.5).astype(int)
    test_rows.append({"model": name, "ROC-AUC": roc_auc_score(y_te, p), "PR-AUC": average_precision_score(y_te, p),
                      "accuracy": accuracy_score(y_te, pred), "precision": precision_score(y_te, pred, zero_division=0),
                      "recall": recall_score(y_te, pred), "F1": f1_score(y_te, pred)})
test_table = pd.DataFrame(test_rows).set_index("model").round(3)
test_table.to_csv(OUT / "02_test_results.csv"); test_table

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
for name, p in probs.items():
    RocCurveDisplay.from_predictions(y_te, p, name=name, ax=ax[0]); PrecisionRecallDisplay.from_predictions(y_te, p, name=name, ax=ax[1])
ax[0].plot([0, 1], [0, 1], "k--", lw=.8); ax[0].set_title("ROC curves (test)"); ax[1].set_title("Precision-recall curves (test)")
ax[1].axhline(y_te.mean(), ls="--", c="grey", lw=.8)
for a in ax: a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "02_roc_pr.png"); plt.show()

**Model choice: Logistic Regression (L1, C = 0.1).**
- All three real models land at a cross-validated ROC-AUC of about **0.64**. The differences are smaller than one standard deviation (about 0.025), and they score about 0.66-0.68 on the test set, against 0.50 for the majority baseline.
- The trees do not find extra non-linear signal, so the simplest, most explainable model is preferred. The L1 penalty also does feature selection.
- **Honest reading:** a ROC-AUC of about 0.65-0.68 means the model ranks a converting view above a non-converting one about two times out of three. That is useful for *prioritising* nudges and campaigns, but not for confident individual decisions. The dataset holds a modest, real signal plus a lot of noise.

### A4. Decision threshold

The threshold is chosen on out-of-fold training predictions, not on the test set.

In [ ]:
final = searches["Logistic Regression"].best_estimator_
oof = cross_val_predict(final, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
ths = np.arange(0.3, 0.71, 0.01); f1s = [f1_score(y_tr, (oof >= t).astype(int)) for t in ths]
best_t = float(ths[int(np.argmax(f1s))])
p = probs["Logistic Regression"]
rows = []
for lab, t in [("default 0.50", 0.5), (f"tuned {best_t:.2f}", best_t)]:
    pred = (p >= t).astype(int)
    rows.append({"threshold": lab, "precision": precision_score(y_te, pred), "recall": recall_score(y_te, pred),
                 "F1": f1_score(y_te, pred), "accuracy": accuracy_score(y_te, pred), "flagged": int(pred.sum())})
thr = pd.DataFrame(rows).set_index("threshold").round(3); thr.to_csv(OUT / "02_threshold.csv")
fig, ax = plt.subplots(figsize=(4.2, 3.6))
ConfusionMatrixDisplay(confusion_matrix(y_te, (p >= best_t).astype(int)), display_labels=["no buy", "buy"]).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Logistic Regression @ {best_t:.2f}"); plt.tight_layout(); plt.savefig(OUT / "02_confusion.png"); plt.show()
thr

### A5. What drives conversion? Permutation importance on raw columns + coefficients

In [ ]:
pi = permutation_importance(final, X_te, y_te, scoring="roc_auc", n_repeats=30, random_state=config.SEED, n_jobs=-1)
imp = pd.DataFrame({"drop_in_ROC_AUC": pi.importances_mean, "std": pi.importances_std}, index=X.columns) \
        .sort_values("drop_in_ROC_AUC", ascending=False)
imp.head(12).round(4).to_csv(OUT / "02_permutation_importance.csv")
names = final.named_steps["prep"].get_feature_names_out()
coef = pd.Series(final.named_steps["clf"].coef_[0], index=[n.split("__", 1)[1] for n in names])
nz = coef[coef != 0].sort_values()
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
imp.head(10)[::-1]["drop_in_ROC_AUC"].plot.barh(ax=ax[0], xerr=imp.head(10)[::-1]["std"], color="#4e79a7")
ax[0].set_title("Permutation importance (test ROC-AUC drop)")
nz.plot.barh(ax=ax[1], color=["#e15759" if v < 0 else "#59a14f" for v in nz]); ax[1].set_title(f"Non-zero L1 coefficients ({len(nz)} of {len(coef)})")
plt.tight_layout(); plt.savefig(OUT / "02_importance.png"); plt.show()
imp.head(8).round(4)

**Interpretation**
- **Membership tier dominates.** Shuffling it costs the most ROC-AUC. Gold and Platinum members convert far more often, so loyalty status is the best single predictor of a purchase.
- **Income band** is second: higher-income shoppers convert more.
- **Product rating** matters: better-rated products convert better. Improving product quality and review collection on low-rated items is a lever.
- **Engagement** (cart + wishlist) adds intent signal.
- **Price and discount have almost no effect on their own.** Conversion barely changes across discount levels overall. The segmentation below shows that the discount response differs by segment, which is why a single global model sees little effect.
- The L1 penalty sets most one-hot brand, colour and state dummies to zero. They carry no reliable signal.

### A6. Did the engineered features help? Ablation with 5-fold CV on train

In [ ]:
eng = ["discount_amount", "price_to_aov", "engagement", "size_missing"]
base_num = [c for c in ml.NUM_FEATURES if c not in eng]; base_bin = [c for c in ml.BIN_FEATURES if c not in eng]
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
def lr_pipe(num, bins):
    pre = ColumnTransformer([("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num),
                             ("bin", SimpleImputer(strategy="most_frequent"), bins),
                             ("cat", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="Unknown")),
                                               ("o", OneHotEncoder(handle_unknown="ignore", min_frequency=10, sparse_output=False))]), ml.CAT_FEATURES)])
    return Pipeline([("prep", pre), ("clf", LogisticRegression(C=0.1, penalty="l1", solver="liblinear", max_iter=3000))])
with_eng = cross_val_score(lr_pipe(ml.NUM_FEATURES, ml.BIN_FEATURES), X_tr, y_tr, cv=cv, scoring="roc_auc")
without = cross_val_score(lr_pipe(base_num, base_bin), X_tr[base_num + base_bin + ml.CAT_FEATURES], y_tr, cv=cv, scoring="roc_auc")
abl = pd.DataFrame({"CV ROC-AUC": [without.mean(), with_eng.mean()], "std": [without.std(), with_eng.std()]},
                   index=["original features only", "+ engineered features"]).round(4)
abl.to_csv(OUT / "02_ablation.csv"); abl

The engineered features change cross-validated ROC-AUC by only a small amount, well inside one standard deviation. They are kept because they make the model easier to read: `engagement` and `discount_amount` are business-friendly. But the main signal comes from the original membership, income and rating columns. This is reported honestly rather than over-claimed.

In [ ]:
import joblib
joblib.dump(final, OUT / "purchase_model.joblib")
json.dump({"model": "Logistic Regression (L1, C=0.1)", "cv_roc_auc": float(cv_table.loc["Logistic Regression", "CV ROC-AUC"]),
           "test": test_table.loc["Logistic Regression"].to_dict(), "threshold": best_t,
           "top_drivers": imp.head(5).index.tolist()}, open(OUT / "supervised_summary.json", "w"), indent=2, default=float)
print("saved purchase_model.joblib + supervised_summary.json")

## Part B: Unsupervised learning, RFM customer segments

**Goal:** give the CRM team a small number of actionable customer groups, and find out *which group responds to discounts*. This answers the assistant question *"Which customer segment seems most price-sensitive?"*.

**Features: RFM**, the standard retail segmentation:
- **Recency:** `days_since_last_purchase`
- **Frequency:** `total_orders`
- **Monetary value:** `avg_order_value`

All three are on very different scales, so they are standardised first. Other attributes (membership, income, prices seen, discount response) are used afterwards to **profile** the segments, not to form them.

*Recall from notebook 01:* `customer_id` is not stable, so segmentation is done on the interaction rows. Each row carries the customer's history as recorded for that interaction.

### B1. Choosing the feature set and k

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
comp = {}
for label, feats in [("RFM (3 features)", ml.SEG_FEATURES), ("wide mix (9 features)", ml.ALT_SEG_FEATURES)]:
    d = ml.kmeans_diagnostics(StandardScaler().fit_transform(df[feats])); d["feature set"] = label; comp[label] = d
diag = pd.concat(comp.values()); diag.round(3).to_csv(OUT / "02_kmeans_diagnostics.csv")
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for label, d in comp.items():
    ax[0].plot(d.k, d.inertia / d.inertia.iloc[0], marker="o", label=label); ax[1].plot(d.k, d.silhouette, marker="o", label=label)
ax[0].set_title("Elbow: inertia (relative to k=2)"); ax[1].set_title("Silhouette score (higher = better separated)")
for a in ax: a.set_xlabel("k"); a.legend()
plt.tight_layout(); plt.savefig(OUT / "02_kmeans_diagnostics.png"); plt.show()
diag.pivot(index="k", columns="feature set", values="silhouette").round(3)

- The **RFM** feature set separates clearly better than the wide mix of 9 features (silhouette about 0.22-0.26 vs about 0.09). The wide mix's features are nearly uncorrelated, so it has no real group structure.
- With RFM, **k = 4** has the highest silhouette, and the elbow flattens after it. Four segments is also a size a marketing team can act on.
- A silhouette of about 0.26 means the segments are *moderately* separated, not sharply distinct groups. This is typical for behavioural data, and it is stated in the conclusions.

In [ ]:
K = int(comp["RFM (3 features)"].sort_values("silhouette", ascending=False).iloc[0]["k"])
km, scaler, Xs, ari = ml.fit_segments(df, K)
profile = ml.segment_profile(df, km.labels_)
names = ml.name_segments(profile)
profile.insert(0, "name", pd.Series(names))
print(f"k = {K} | silhouette = {silhouette_score(Xs, km.labels_):.3f} | agreement with Ward hierarchical clustering (ARI) = {ari:.2f}")
profile.to_csv(OUT / "02_segment_profile.csv")
profile[["name", "size", "share", "days_since_last_purchase", "total_orders", "avg_order_value", "membership_ord",
         "income_ord", "purchase_rate", "return_rate", "conv_deep_discount", "conv_low_discount", "discount_lift"]]

In [ ]:
from sklearn.decomposition import PCA
pc = PCA(2, random_state=config.SEED).fit_transform(Xs)
lab = pd.Series(km.labels_).map(names)
fig, ax = plt.subplots(1, 3, figsize=(19, 4.8))
sns.scatterplot(x=pc[:, 0], y=pc[:, 1], hue=lab, s=12, ax=ax[0], alpha=.7); ax[0].set_title("Segments in PCA space (RFM)"); ax[0].legend(title="segment", fontsize=8)
sns.scatterplot(data=df.assign(segment=lab), x="total_orders", y="avg_order_value", hue="segment", s=12, ax=ax[1], alpha=.7, legend=False)
ax[1].set_title("Frequency vs monetary value")
p2 = profile.set_index("name")
x = np.arange(len(p2)); ax[2].bar(x - .2, p2.conv_low_discount, .4, label="conversion, discount <= 10%")
ax[2].bar(x + .2, p2.conv_deep_discount, .4, label="conversion, discount >= 25%")
ax[2].set_xticks(x); ax[2].set_xticklabels(p2.index, rotation=15, fontsize=8); ax[2].legend(fontsize=8); ax[2].set_title("Price sensitivity by segment")
plt.tight_layout(); plt.savefig(OUT / "02_segments.png"); plt.show()
ml.save_segment_summary(profile.drop(columns="name"), names)
print("saved outputs/segment_summary.json (used by the assistant's segment_insights tool)")

### B2. The segments in business language
- **High-Value Regulars:** the highest average order value (about ₹4,800), recent and fairly frequent. Protect them with service, early access and loyalty perks. Discounts move them only a little.
- **Frequent Small-Basket Buyers:** the most orders (about 11) but the smallest baskets. They are the **most price-sensitive** group: conversion rises from about 48% on items with ≤10% discount to about 61% on items with ≥25% discount (+14 points). Targeted coupons and bundles that raise basket size should work best here.
- **Lapsing Customers:** about 70 days since the last purchase, against about 28 days for the others. Win-back campaigns (reminders, a time-limited offer) are the priority.
- **Occasional Buyers:** few orders (about 4) and a mid-sized basket. They barely react to discounts (+1 point), so content, reviews and recommendations matter more than price.

Membership, income and the prices seen are similar across segments. RFM captures *behaviour*, which is a different axis from the *profile* that drives conversion in Part A. Together they give the business two complementary views.

**Validation:**
- The silhouette score and elbow (above).
- Cluster sizes are balanced (17-29%).
- Ward hierarchical clustering on the same data only partly agrees with K-Means (ARI shown above). This confirms that the boundaries are soft, so the segments should be read as *tendencies*, not hard customer types.